
# Vertex AI Project 1 — Hello Vertex: Deploy a Tiny Model and Predict

This notebook walks you through:
1) Training a tiny scikit-learn model locally  
2) Uploading the model to **Vertex AI Model Registry**  
3) Deploying the model to a **Vertex AI Endpoint** (online predictions)  
4) Sending **online** and **batch** predictions  
5) (Optional) Enabling basic **logging**  
6) (Optional) Cleanup resources (undeploy endpoint, delete endpoint)

> **Before you start**: Make sure you have a Google Cloud project with billing enabled and the Vertex AI API enabled.



## 0) Prerequisites
- Install the Vertex AI Python SDK
- Authenticate with Google Cloud (in Colab: `gcloud auth application-default login`)
- Set your **PROJECT_ID**, **LOCATION/REGION**, and **GCS staging bucket**.


In [ ]:
!python -m pip install --upgrade pip
!python -m pip install packaging setuptools wheel
!python -m pip install -U google-cloud-aiplatform scikit-learn joblib google-cloud-storage

: 

In [ ]:
!pip install --upgrade pip
!pip install packaging setuptools wheel
!pip -q install -U google-cloud-aiplatform scikit-learn joblib google-cloud-storage

: 


## 1) Configure your environment
Edit the variables below to match your setup:


In [ ]:

from google.cloud import aiplatform
from google.cloud import storage
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import joblib, os, json, time, uuid

# === EDIT THESE ===
PROJECT_ID = "instr-cs795-fall25-hqin-1"
LOCATION   = "us-east4"      # e.g., "us-central1", "us-east5", "europe-west4"
BUCKET     = "gs://instr-cs795-fall25-hqin-1-arasm002"  # must exist already (no trailing slash)

# Derived/utility
TIMESTAMP = time.strftime("%Y%m%d-%H%M%S")
DISPLAY_NAME = f"iris-rf-sklearn-{TIMESTAMP}"
MODEL_DIR  = f"model-artifacts-{TIMESTAMP}"
MODEL_FILE = f"{MODEL_DIR}/model.pkl"
BATCH_INPUT_DIR = f"batch_inputs_{TIMESTAMP}"
GCS_INPUT_PREFIX = f"{BUCKET}/batch_inputs/{TIMESTAMP}/"
GCS_OUTPUT_PREFIX = f"{BUCKET}/batch_outputs/{TIMESTAMP}/"

print("Project:", PROJECT_ID)
print("Location:", LOCATION)
print("Bucket:", BUCKET)


: 


## 2) Train a tiny model locally
We'll train a small RandomForest on the Iris dataset and save it as a `.pkl`.


In [ ]:

# Train a toy model
X, y = load_iris(return_X_y=True)
clf = RandomForestClassifier(n_estimators=10, random_state=0).fit(X, y)

# Save artifacts locally
os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(clf, MODEL_FILE)
print("Saved model to:", MODEL_FILE)



## 3) Initialize Vertex AI
Initialize the SDK with your project, region, and a staging bucket.


In [ ]:

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET)
print("Initialized Vertex AI")



## 4) Upload model to Vertex AI Model Registry
We use the prebuilt **scikit-learn** prediction container so you don't have to write any serving code.


In [ ]:

SKLEARN_URI = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-4:latest"

model = aiplatform.Model.upload(
    display_name=DISPLAY_NAME,
    artifact_uri=MODEL_DIR,                      # SDK will upload this folder to GCS for you
    serving_container_image_uri=SKLEARN_URI,
)
model.wait()
print("Model uploaded. Resource name:", model.resource_name)



## 5) Create an Endpoint and Deploy the model


In [ ]:

endpoint = aiplatform.Endpoint.create(display_name=f"iris-endpoint-{TIMESTAMP}")
endpoint.wait()
print("Endpoint created:", endpoint.resource_name)

deployed_model = model.deploy(
    endpoint=endpoint,
    machine_type="n1-standard-2",
    traffic_split={"0": 100},
)
print("Model deployed.")
print("Endpoint:", endpoint.resource_name)



## 6) Online prediction
Send a couple of instances to the endpoint. We reuse `X` from the Iris dataset.


In [ ]:

instances = [X[0].tolist(), X[1].tolist()]
prediction = endpoint.predict(instances=instances)
print("Predictions:", prediction.predictions)



## 7) (Optional) Batch prediction
Batch prediction reads from **GCS** and writes outputs back to **GCS**.

### 7.1 Create a small JSONL input set and upload to GCS
Each line in JSONL is one instance: `{"instances": [<features>]}` for scikit-learn prebuilt container.


In [ ]:

# Create local batch input directory
os.makedirs(BATCH_INPUT_DIR, exist_ok=True)

# Write a tiny JSONL file
batch_file = os.path.join(BATCH_INPUT_DIR, "inputs.jsonl")
with open(batch_file, "w") as f:
    for i in range(5):
        payload = {"instances": X[i].tolist()}
        f.write(json.dumps(payload) + "\n")
print("Wrote batch inputs to:", batch_file)

# Upload to GCS
client = storage.Client(project=PROJECT_ID)
bucket_name = BUCKET.replace("gs://", "").split("/")[0]
prefix = "/".join(BUCKET.replace("gs://", "").split("/")[1:])  # optional subpath
gcs_dir = f"batch_inputs/{TIMESTAMP}/"
if prefix:
    gcs_dir = f"{prefix.rstrip('/')}/{gcs_dir}"

bucket = client.bucket(bucket_name)
blob = bucket.blob(f"{gcs_dir}inputs.jsonl")
blob.upload_from_filename(batch_file)
print(f"Uploaded to gs://{bucket_name}/{gcs_dir}inputs.jsonl")

print("GCS input prefix:", f"gs://{bucket_name}/{gcs_dir}")
print("GCS output prefix:", GCS_OUTPUT_PREFIX)



### 7.2 Launch batch prediction job


In [ ]:

bp_job = model.batch_predict(
    job_display_name=f"iris-batch-{TIMESTAMP}",
    gcs_source=[f"gs://{bucket_name}/{gcs_dir}*.jsonl"],
    gcs_destination_prefix=GCS_OUTPUT_PREFIX,
    instances_format="jsonl",
    predictions_format="jsonl",
)
print("Batch prediction job started:", bp_job.resource_name)
bp_job.wait()
print("Batch prediction job completed.")



## 8) (Optional) Enable logging & view logs
Vertex AI sends container logs to **Cloud Logging**. You can also enable **access logging** from the console.
- In Cloud Console, go to **Logging → Logs Explorer** and filter on your endpoint or model resource name.
- For access logging (request/response metadata), enable it on the Endpoint's **Monitoring** tab.



## 9) (Optional) Cleanup
Undeploy the model and delete the endpoint to stop charges when you're done.


In [ ]:

# Uncomment to cleanup when finished:
# endpoint.undeploy_all()
# endpoint.delete()
# print("Endpoint undeployed and deleted.")
